In [2]:
import sys
import os


# sys.path.append('..')
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import numpy as np
import pandas as pd


print(os.getcwd())
import datetime
from src.models.cmae import CMAE
from src.data.load_cifar10 import get_cifar10_loaders
from src.data.load_cifar100 import get_cifar100_loaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
import matplotlib.pyplot as plt


c:\Users\aczar\Desktop\polibuda\ML_projekt\src\notebooks_test_train
cuda


In [3]:
# Dane
train_loader, val_loader, test_loader = get_cifar10_loaders(batch_size=64)
# train_loader, val_loader, test_loader = get_cifar100_loaders(batch_size=64)

In [4]:
# Model
model = CMAE(latent_dim=256).to(device)

# optimizer
# optimizer = optim.Adam(model.parameters(), lr=0.001)


optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.05) # AdamW niby lepszy dla contrastive learning

In [5]:
# Trening

BASE_DIR = os.getcwd()
save_dir = os.path.join(BASE_DIR, '..', 'training_results', 'cmae', 'cifar10')
# save_dir = os.path.join(BASE_DIR, '..', 'training_results', 'cmae', 'cifar100')
print(save_dir)

writer_dir = os.path.join(BASE_DIR, '..', 'training_results', 'tb_cmae', 'cifar10')
# writer_dir = os.path.join(BASE_DIR, '..', 'training_results', 'tb_cmae', 'cifar100')

os.makedirs(save_dir, exist_ok=True)
os.makedirs(writer_dir, exist_ok=True)


timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
writer = SummaryWriter(log_dir=os.path.join(writer_dir, f'cmae_{timestamp}'))

num_epochs = 50


history = {
    'train_loss': [], 'train_rec': [], 'train_con': [],
    'val_loss': [], 'val_rec': [], 'val_con': []
}

epoch_number = 0
best_val_loss = float('inf')

print("Starting training...")

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_rec_loss = 0.0
    train_con_loss = 0.0 

    for batch_idx, (image, _) in enumerate(train_loader):
        image = image.to(device)

        outputs = model(image)
        # loss - zmae daje 3 wartosci
        loss, rec_loss, con_loss = model.compute_loss(outputs)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # update nauczyciela

        model.update_target()

        train_loss += loss.item()
        train_rec_loss += rec_loss.item()
        train_con_loss += con_loss.item()

        if batch_idx % 100 == 0:
            avg_loss = train_loss / (batch_idx + 1)
            avg_rec = train_rec_loss / (batch_idx + 1)
            avg_con = train_con_loss / (batch_idx + 1)

            print(f"  [{epoch+1}/{num_epochs}] Batch {batch_idx}/{len(train_loader)} "
                  f"Loss: {avg_loss:.4f} (REC: {avg_rec:.4f}, CON: {avg_con:.4f})")
    
    avg_train_loss = train_loss / len(train_loader)
    avg_train_rec = train_rec_loss / len(train_loader)
    avg_train_con = train_con_loss / len(train_loader)

    history['train_loss'].append(avg_train_loss)
    history['train_rec'].append(avg_train_rec)
    history['train_con'].append(avg_train_con)

    writer.add_scalar('Loss/Train', avg_train_loss, epoch_number)
    writer.add_scalar('Reconstruction_Loss/Train', avg_train_rec, epoch_number)
    writer.add_scalar('Contrastive_Loss/Train', avg_train_con, epoch_number)
    epoch_number += 1


    # validation
    model.eval()
    val_loss = 0.0
    val_rec_loss = 0.0
    val_con_loss = 0.0
    all_outputs = []

    with torch.no_grad():
        for image, _ in val_loader:
            image = image.to(device)

            outputs = model(image)
            loss, rec_loss, con_loss = model.compute_loss(outputs)

            val_loss += loss.item()
            val_rec_loss += rec_loss.item()
            val_con_loss += con_loss.item()

            all_outputs.append(outputs)

    if epoch % 5 == 0:
        outputs = all_outputs[0] 
        # wyciąganie danych do wizualizacji
        reconstructed = outputs['reconstructed_image']

        x_original, _, mask = outputs['loss_recon'] # oryginał i maska

        n_images = min(8, image.size(0))

        writer.add_images('Original', x_original[:n_images], epoch)
        writer.add_images('Reconstructed', reconstructed[:n_images], epoch)

        # wizualizacja maski
        mask_vis = mask[:n_images].repeat(1, 3, 1, 1)  # powielenuie kanały do 3
        writer.add_images('Mask', mask_vis, epoch)

        # wizualizacja zakodowanych reprezentacji -- tak widzial student
        masked_input = x_original[:n_images] * mask[:n_images]
        writer.add_images('Masked_Input', masked_input, epoch)

    avg_val_loss = val_loss / len(val_loader)
    avg_val_rec = val_rec_loss / len(val_loader)
    avg_val_con = val_con_loss / len(val_loader)

    history['val_loss'].append(avg_val_loss)
    history['val_rec'].append(avg_val_rec)
    history['val_con'].append(avg_val_con)

    writer.add_scalar('Loss/val_total', avg_val_loss, epoch)
    writer.add_scalar('Loss/val_rec', avg_val_rec, epoch)
    writer.add_scalar('Loss/val_con', avg_val_con, epoch)

    print(f"Epoch [{epoch+1}/{num_epochs}] ")
    print(f"  Train Loss: {avg_train_loss:.4f} (REC: {avg_train_rec:.4f}, CON: {avg_train_con:.4f})")
    print(f"  Val   Loss: {avg_val_loss:.4f} (REC: {avg_val_rec:.4f}, CON: {avg_val_con:.4f})")

    # zapisywanie najlepszego modelu
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        checkpoint ={

            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss,
            'latent_dim': 256,
            'train_losses': history['train_loss'],
            'val_losses': history['val_loss'],
            
        }
        timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        checkpoint_path = os.path.join(save_dir, f'cmae_cifar10_best_{timestamp}.pt')
        # checkpoint_path = os.path.join(save_dir, f'cmae_cifar100_best_{timestamp}.pt')
        torch.save(checkpoint, checkpoint_path)

# zapisywanie historii treningu
df = pd.DataFrame({
    'epoch': range(1, num_epochs + 1),
    'train_loss': history['train_loss'],
    'val_loss': history['val_loss'],
    'train_rec': history['train_rec'],
    'val_rec': history['val_rec'],
    'train_con': history['train_con'],
    'val_con': history['val_con']


})

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
history_csv = os.path.join(save_dir, f'cmae_cifar10_training_results_{timestamp}.csv')
# history_csv = os.path.join(save_dir, f'cmae_cifar100_training_results_{timestamp}.csv')
df.to_csv(history_csv, index=False)

c:\Users\aczar\Desktop\polibuda\ML_projekt\src\notebooks_test_train\..\training_results\cmae\cifar10
Starting training...
  [1/50] Batch 0/625 Loss: 2.5731 (REC: 0.0574, CON: 5.0314)
  [1/50] Batch 100/625 Loss: 0.9539 (REC: 0.0320, CON: 1.8438)
  [1/50] Batch 200/625 Loss: 0.7279 (REC: 0.0304, CON: 1.3950)
  [1/50] Batch 300/625 Loss: 0.6120 (REC: 0.0294, CON: 1.1652)
  [1/50] Batch 400/625 Loss: 0.5442 (REC: 0.0287, CON: 1.0309)
  [1/50] Batch 500/625 Loss: 0.5034 (REC: 0.0284, CON: 0.9501)
  [1/50] Batch 600/625 Loss: 0.4718 (REC: 0.0281, CON: 0.8874)
Epoch [1/50] 
  Train Loss: 0.4646 (REC: 0.0280, CON: 0.8731)
  Val   Loss: 0.3591 (REC: 0.0263, CON: 0.6657)
  [2/50] Batch 0/625 Loss: 0.2793 (REC: 0.0246, CON: 0.5094)
  [2/50] Batch 100/625 Loss: 0.2859 (REC: 0.0257, CON: 0.5203)
  [2/50] Batch 200/625 Loss: 0.2756 (REC: 0.0257, CON: 0.4996)
  [2/50] Batch 300/625 Loss: 0.2623 (REC: 0.0255, CON: 0.4737)
  [2/50] Batch 400/625 Loss: 0.2551 (REC: 0.0254, CON: 0.4594)
  [2/50] Batch 5